# Agente SQL: explicações com evidências

> **Dados integralmente sintéticos.** Execute `python -m src.pipeline` antes deste notebook.

O agente usa o SDK OpenAI com ferramenta de consultas SQL auditadas. Sem chave, fornece narrativa determinística. O modo online é opt-in; o notebook roda offline.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))
import pandas as pd
from IPython.display import display, Image
from src.config import DATA, REPORTS, FIGURES


## Catálogo de consultas restritas

In [2]:
from src.agent import SQL, explain
for name, query in SQL.items():
    print(name, query)

executivo SELECT count(*) AS transactions, round(sum(amount), 2) AS total_amount, round(avg(amount), 2) AS average_ticket FROM transactions
anomalias SELECT transaction_id, account_id, date, amount, hour, anomaly_score FROM anomaly_cases ORDER BY anomaly_score DESC LIMIT 5
churn SELECT account_id, churn_probability, recency, count_recent, count_previous, activity_ratio FROM churn_predictions ORDER BY churn_probability DESC LIMIT 5


## Anomalias e churn em linguagem natural

In [3]:
for question in ['Explique as anomalias', 'Explique o risco de churn', 'Resumo executivo']:
    result = explain(question)
    print(question, '\n', result['answer'], '\n')
    display(result['evidence'])

Explique as anomalias 
 A transação 2102689 da conta 448 lidera o ranking de anomalias: R$ 4656.20, às 6h, score 0.845. O detector considera valor, horário e dia da semana. Um score elevado exige revisão; não comprova fraude. 



[{'report': 'anomalias',
  'sql': 'SELECT transaction_id, account_id, date, amount, hour, anomaly_score FROM anomaly_cases ORDER BY anomaly_score DESC LIMIT 5',
  'rows': [{'transaction_id': 2102689,
    'account_id': 448,
    'date': '2025-12-28',
    'amount': 4656.2,
    'hour': 6,
    'anomaly_score': 0.8453968405649634},
   {'transaction_id': 1912991,
    'account_id': 26,
    'date': '2025-10-26',
    'amount': 2165.5,
    'hour': 1,
    'anomaly_score': 0.8429593813243326},
   {'transaction_id': 2008779,
    'account_id': 3232,
    'date': '2025-11-26',
    'amount': 2943.91,
    'hour': 0,
    'anomaly_score': 0.8413383140349918},
   {'transaction_id': 2007447,
    'account_id': 1097,
    'date': '2025-11-26',
    'amount': 3817.32,
    'hour': 3,
    'anomaly_score': 0.8413383140349918},
   {'transaction_id': 2007621,
    'account_id': 1380,
    'date': '2025-11-26',
    'amount': 3426.03,
    'hour': 3,
    'anomaly_score': 0.8413383140349918}]}]

Explique o risco de churn 
 A conta 4815 tem probabilidade prevista de inatividade em 30 dias de 98.9%. Realizou 1 transações nos últimos 30 dias, ante 9 na janela anterior, e está há 29 dias sem transacionar. São sinais associados ao risco; não provam a causa do abandono. O ranking cobre somente contas do teste reservado. 



[{'report': 'churn',
  'sql': 'SELECT account_id, churn_probability, recency, count_recent, count_previous, activity_ratio FROM churn_predictions ORDER BY churn_probability DESC LIMIT 5',
  'rows': [{'account_id': 4815,
    'churn_probability': 0.9894338084979376,
    'recency': 29,
    'count_recent': 1,
    'count_previous': 9,
    'activity_ratio': 0.2},
   {'account_id': 4057,
    'churn_probability': 0.9815969273795592,
    'recency': 25,
    'count_recent': 1,
    'count_previous': 17,
    'activity_ratio': 0.1111111111111111},
   {'account_id': 860,
    'churn_probability': 0.9703110389835048,
    'recency': 25,
    'count_recent': 2,
    'count_previous': 13,
    'activity_ratio': 0.2142857142857142},
   {'account_id': 2379,
    'churn_probability': 0.9655468469971268,
    'recency': 24,
    'count_recent': 3,
    'count_previous': 11,
    'activity_ratio': 0.3333333333333333},
   {'account_id': 815,
    'churn_probability': 0.964712236346282,
    'recency': 24,
    'count_rece

Resumo executivo 
 A base contém 2,115,556 transações e R$ 188,857,774.06 movimentados, com ticket médio de R$ 89.27. 



[{'report': 'executivo',
  'sql': 'SELECT count(*) AS transactions, round(sum(amount), 2) AS total_amount, round(avg(amount), 2) AS average_ticket FROM transactions',
  'rows': [{'transactions': 2115556,
    'total_amount': 188857774.06,
    'average_ticket': 89.27}]}]

## Contrato de segurança

In [4]:
try:
    from src.agent import query_database
    query_database('DROP TABLE accounts')
except ValueError as error:
    print('Entrada recusada:', error)

Entrada recusada: Relatório não permitido.


## Leitura crítica

O banco é aberto em modo somente leitura com acesso externo desativado. A ferramenta aceita apenas três identificadores de relatório; não executa SQL inventado pelo modelo. Perguntas arbitrárias não são suportadas pelo fallback. Chaves vêm do ambiente, nunca de arquivos versionados. As evidências dos rankings são sintéticas. O modo com API tem teste de protocolo simulado; uma chamada real depende de credenciais e não integra a reprodução offline.